In [4]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from PIL import ImageFile

# ==============================
# Step 1 — Load Dataset
# ==============================
print("Loading dataset...")

data_dir = "../../data sheets/training_set"
ImageFile.LOAD_TRUNCATED_IMAGES = True

batch_size = 32
img_size = (224, 224)

datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=25,
    width_shift_range=0.25,
    height_shift_range=0.25,
    shear_range=0.2,
    zoom_range=0.3,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)

train_gen = datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="sparse",
    subset='training',
    shuffle=True
)

val_gen = datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="sparse",
    subset='validation',
    shuffle=False
)

# ==============================
# Step 2 — Fine-tune ResNet50
# ==============================
print("Building ResNet50 base model...")

base_model = ResNet50(weights="imagenet", include_top=False, input_shape=(224, 224, 3))

# Freeze most layers, unfreeze last few
for layer in base_model.layers[:-20]:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.4)(x)
predictions = Dense(5, activation='softmax')(x)

ft_model = Model(inputs=base_model.input, outputs=predictions)
ft_model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

print("Fine-tuning ResNet50...")
ft_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=5,
    callbacks=[early_stop],
    verbose=1
)

# ==============================
# Step 3 — Extract Features
# ==============================
print("Extracting features from fine-tuned ResNet50...")

feature_extractor = Model(inputs=ft_model.input, outputs=ft_model.get_layer("conv5_block3_out").output)

def extract_features(generator, model):
    features, labels = [], []
    for i in range(len(generator)):
        try:
            x_batch, y_batch = generator[i]
            feat_batch = model.predict(x_batch, verbose=0)
            feat_batch = feat_batch.reshape(feat_batch.shape[0], -1)
            features.append(feat_batch)
            labels.append(y_batch)
            if (i + 1) * batch_size >= generator.n:
                break
        except Exception as e:
            print(f"Skipping batch {i} due to error: {e}")
            continue
    X = np.vstack(features)
    y = np.hstack(labels)
    return X, y

X_train, y_train = extract_features(train_gen, feature_extractor)
X_test, y_test = extract_features(val_gen, feature_extractor)

print("Feature extraction complete.")
print("Train features:", X_train.shape, "Test features:", X_test.shape)

# ==============================
# Step 4 — Normalize + PCA
# ==============================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

pca = PCA(n_components=256, random_state=42)
X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)

# ==============================
# Step 5 — Train SVM
# ==============================
print("Performing Grid Search for best SVM parameters...")

param_grid = {
    'C': [1, 5, 10, 50],
    'gamma': ['scale', 0.001, 0.0005],
    'kernel': ['rbf']
}

grid = GridSearchCV(SVC(random_state=42), param_grid, cv=3, n_jobs=-1, verbose=2)
grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
clf = grid.best_estimator_

print("Training final SVM...")
clf.fit(X_train, y_train)

# ==============================
# Step 6 — Evaluate
# ==============================
print("\nEvaluating model...")
y_pred = clf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("\nAccuracy:", round(accuracy * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Loading dataset...
Found 4000 images belonging to 5 classes.
Found 1000 images belonging to 5 classes.
Building ResNet50 base model...
Fine-tuning ResNet50...


c:\Users\MODERN\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 463s 4s/step - accuracy: 0.2849 - loss: 1.6890 - val_accuracy: 0.4220 - val_loss: 1.3778
Epoch 2/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 571s 5s/step - accuracy: 0.4633 - loss: 1.3265 - val_accuracy: 0.4780 - val_loss: 1.2812
Epoch 3/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 612s 5s/step - accuracy: 0.5324 - loss: 1.1787 - val_accuracy: 0.4810 - val_loss: 1.2420
Epoch 4/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 516s 4s/step - accuracy: 0.5859 - loss: 1.0725 - val_accuracy: 0.5020 - val_loss: 1.2505
Epoch 5/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 424s 3s/step - accuracy: 0.6243 - loss: 0.9649 - val_accuracy: 0.4620 - val_loss: 1.3212
Extracting features from fine-tuned ResNet50...
Feature extraction complete.
Train features: (4000, 100352) Test features: (1000, 100352)
Performing Grid Search for best SVM parameters...
Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best Parameters: {'C': 1, 'gamma': 'scale', 'kernel': 'rbf'}
Training final SVM...

Evaluating model...

Accura